In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load your API key from .env
load_dotenv()

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "What is the capital of the United States?"}]
)

print(response.choices[0].message.content)


The capital of the United States is Washington, D.C.


In [ ]:
# x-axis: dose of morphine, y-axis: inferred conversion coefficient
# x-axis: dose or morphine, y-axis: ratio of recommended opioid to Nielson recommendation

# DATAFRAME:
    ## PROMPT includes NIELSON conversion coefficient
    ## PROMPT includes EXAMPLES
    ## ADMINISTRATION_METHOD
    ## SUBSTANCE
    ## DOSE_FROM (random based on 10 mg to 120 mg oral morphine)
    ## EXPECTED DOSE_TO
    ## OBSERVED DOSE_TO FROM LLM
    ## RATIO OBSERVED/EXPECTED

# AGGREGATES:
    ## PROMPT includes NIELSON conversion coefficient
    ## PROMPT includes EXAMPLES
    ## ADMINISTRATION_METHOD
    ## SUBSTANCE
    ## PROPORTION of RATIO outside of 0.7 - 1.3

    # p-value for a non-zero slope
    # p-value for a non-one mean

    # CDF of RATIO EXPECTED/OBSERVED by including Nielson, including examples

In [ ]:
# helper function
# generate a random dose of opioid based on OME
# choose a random value on a log scale between 10 mg and 120 mg
# use Neilson 2015 to convert from OME to DOSE_FROM

In [ ]:
# prompt function
# generate a prompt from inputs:
    # INCLUDE NIELSON?
    # INCLUDE 10 mg and 120 mg EXAMPLES?
    # ADMINISTRATION_METHOD

In [ ]:
# an OPIOID_FROM and an OPIOID_TO both have a ADMINISTRATION_METHOD and SUBSTANCE
# ask to convert from DOSE_FROM of OPIOID_FROM to OPIOID_TO
# include a conversion coefficient indicating the oral morphine equivalent

In [ ]:
# Create a prompted_conversion object equivalent to a row of the dataframe
# has properties:
                
    ## ADMINISTRATION_METHOD_FROM
    ## SUBSTANCE_FROM
        ## DOSE_FROM (random based on 10 mg to 120 mg oral morphine)

    ## ADMINISTRATION_METHOD_TO (default oral)
    ## SUBSTANCE_TO (default morphine)
        ## EXPECTED DOSE_TO ACCORDING TO NIELSON
            ## PROMPT includes NIELSON conversion coefficient
            ## PROMPT includes EXAMPLES
                # PROMPT
                # MODEL PROMPTED
                    # RESPONSE
                    # OBSERVED DOSE_TO
                    # RATIO OBSERVED/EXPECTED
                    # RATIO INSIDE 0.7 - 1.3

In [ ]:
# Create an experiment object
    # INPUT:
        # TUPLES of (FROM, TO) or a LIST of FROM, defaulting to morphine
        # NUMBER of REPETITIONS
        # INCLUDE NIELSON?
        # INCLUDE 10 mg and 120 mg EXAMPLES?
    # OUTPUT:
        # LIST of prompted_conversions
        # DATAFRAME of prompted_conversions
        # AGGREGATES
            # by (FROM, TO) and total
                # Regression of EXPECTED DOSE vs. RATIO OBSERVED/EXPECTED
                # p-value non-zero slope
                # p-value non-one mean
            # by (FROM, TO) and total
                # CDF of RATIO OBSERVED/EXPECTED
            # by (FROM, TO) and total
                # Proportion RATIO INSIDE 0.7 - 1.3            
        # FIGURES
            # Scatter plot of EXPECTED DOSE v. RATIO OBSERVED/EXPECTED by (FROM, TO)
                # Should we include a regression line?
            # CDF of RATIO OBSERVED/EXPECTED by (FROM, TO)
            # Proportion of RATIO INSIDE 0.7 - 1.3 by (FROM, TO)



In [2]:
# Conversion Service
from opioid import Opioid
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS

conversion_service = ConversionService(NIELSON_CONVERSIONS)

iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone")
oral_morphine = Opioid(administration_method="oral", substance="morphine")

random_dose = conversion_service.generate_random_dose(iv_fentanyl)
print(f"Random dose for {iv_fentanyl.administration_method}_{iv_fentanyl.substance}: {random_dose:.2f} {iv_fentanyl.unit}")

equivalent_dose_oral_morphine = conversion_service.calculate_expected_dose(random_dose, iv_fentanyl, oral_morphine)
print(f"Equivalent dose for {oral_morphine.administration_method}_{oral_morphine.substance}: {equivalent_dose_oral_morphine:.2f} {oral_morphine.unit}")

Random dose for iv_fentanyl: 0.30 mcg/hour
Equivalent dose for oral_morphine: 29.58 mg/day


In [3]:
print(iv_fentanyl)

Opioid(administration_method='iv', substance='fentanyl', unit='mcg/hour')


In [ ]:
# Prompt Service
from prompt_service import PromptService

# Initialize the ConversionService
conversion_service = ConversionService(NIELSON_CONVERSIONS)

# Initialize the PromptService with the ConversionService instance
prompt_service = PromptService(conversion_service)

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/hour")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone", unit="mg/day")
oral_morphine = Opioid(administration_method="oral", substance="morphine", unit="mg/day")

# Generate a random dose for the source opioid
dose_from = conversion_service.generate_random_dose(iv_fentanyl)

# Generate a prompt
prompt = prompt_service.generate_prompt(
    from_opioid=iv_fentanyl,
    to_opioid=oral_morphine,
    from_dose=dose_from,
    include_nielson=False,
    include_examples=True
)

# Print the generated prompt
print("Generated Prompt:")
print(prompt)

Generated Prompt:
Convert 0.12 mcg/hour of fentanyl (iv) to mg/day of morphine (oral).
How many mg/day of morphine (oral) is equivalent to 0.12 mcg/hour of fentanyl (iv)?

100 OME for fentanyl (iv) means 10 mg/day of morphine (oral) is equivalent to 0.10 mcg/hour of fentanyl (iv).
And 120 mg/day of morphine (oral) is equivalent to 1.20 mcg/hour of fentanyl (iv).

